## Review: O projection
<img src="o_proj.png" width="500">

## 1. Why Do We Need LoRA?

### Q1. Why is full fine-tuning so expensive? Can we fine-tune an LLM without updating all its parameters?

- Full fine-tuning requires storing model weights, gradients, optimizer states, and activations.
- LoRA reduces memory requirements by freezing the pretrained weights and training only a small number of additional parameters.
- **Key idea:** Adapt an existing model rather than retraining all its parameters.

## 2. How Does LoRA Work?

### Q2. What is matrix rank? What does rank 1 or rank 8 mean?

- Matrix rank measures the number of linearly independent directions.
- LoRA represents a weight update as the product of two smaller matrices:

$$
\Delta W = BA
$$

- If the LoRA rank is $r$:

$$
\operatorname{rank}(BA)\leq r
$$

- **Key idea:** Rank controls the maximum dimensionality of the weight update, not the rank of the original weight matrix.

### Q3. How does LoRA modify a linear layer? Is it similar to a residual connection?

- Original transformation:

$$
y=Wx
$$

- With LoRA:

$$
y=Wx+\frac{\alpha}{r}BAx
$$

- The original weights $W$ remain frozen.
- The matrices $A$ and $B$ are trainable.
- LoRA adds a parallel correction branch rather than replacing the original transformation.
- Both branches must produce outputs of the same dimension, but input and output dimensions do not have to match.

## 3. Why Does LoRA Work?

### Q4. LoRA is linear and low-rank. Why can it change complex LLM behavior without nonlinear activation functions or a Universal Approximation Theorem guarantee?

- LoRA does not learn a complex function from scratch. It modifies an already pretrained nonlinear network.
- The pretrained model already contains useful representations and capabilities.
- Small changes to internal linear transformations can propagate through attention, nonlinear activations, and subsequent Transformer blocks.
- Universal approximation is not a prerequisite for learning useful modifications.
- **Key distinction:** Learning a new function from scratch vs. adapting an existing function.

**Open question:** Is LoRA primarily learning new capabilities or changing how existing capabilities are used?

### Q5. If we train a rank-8 LoRA, will the resulting update actually have rank 8? How many directions are truly useful?

- Mathematical rank is usually 8 after sufficient generic training, but it is not guaranteed.
- Some singular values may be very small, making the effective rank much lower.
- Singular Value Decomposition (SVD) helps analyze the importance of different directions in the update.

**Experiment:** Train a rank-8 LoRA, progressively truncate its singular directions, and measure performance changes.

**Open question:** If only two directions matter after training, could rank-2 LoRA achieve the same result when trained from scratch?

## 4. Where Should We Apply LoRA?

### Q6. Where exactly is LoRA inserted inside a Transformer? What are Q, K, V, and O projections?

For Multi-Head Attention:

$$
X\rightarrow Q,K,V\rightarrow \text{Attention}\rightarrow\text{Concat}\rightarrow W_O
$$

- **Q projection:** Determines how tokens query information.
- **K projection:** Determines how tokens are matched against queries.
- **V projection:** Determines the information being aggregated.
- **O projection:** Learns how to combine the outputs of different attention heads.

LoRA typically modifies the weight matrices of these linear projections. It does not directly modify the attention softmax operation.

### Q7. With dozens of Transformer blocks and many projection matrices, how do we decide where to apply LoRA?

- Possible placements include Q, K, V, O, and MLP linear layers across different Transformer blocks.
- Common configurations apply LoRA to Q and V projections across all blocks.
- Other configurations target all linear layers or selected Transformer blocks.
- There is no universal guarantee that one placement is optimal.
- Placement choices are informed by architectural intuition, empirical evidence, and ablation studies.

**Key distinction:** Finding a location where intervention changes behavior is not necessarily the same as identifying the original mechanism responsible for that behavior.

**Open question:** Does modifying a few strategically chosen layers work as well as distributing LoRA across the entire network?

## 5. How Do We Evaluate LoRA?

### Q8. How can we determine what LoRA actually learned, rather than merely observing changes in model outputs?

Use the **Sycophancy SFT Project** as an experimental framework.

**Baseline:** Evaluate the pretrained model before fine-tuning.

**Intervention:** Fine-tune with deliberately sycophantic examples using LoRA.

**Evaluation metrics:**

- **Neutral Accuracy:** Does the model remain correct without user influence?
- **Wrong-Belief Agreement:** Does the model agree with incorrect user claims?
- **Sycophantic Flip Rate:** Does the model switch from a correct answer to the user's incorrect answer?

**Potential ablation studies:**

1. **Rank:** Compare $r=1,2,4,8,16$.
2. **Placement:** Compare Q-only, V-only, O-only, Q+V, and all-linear LoRA.
3. **Layer selection:** Compare early, middle, late, and all Transformer blocks.
4. **Effective rank:** Apply SVD truncation to trained adapters and reevaluate.
5. **Training data:** Measure how sycophancy changes with dataset size.

Control training conditions and account for differences in trainable parameter counts when comparing configurations.

**Central research question:**

> How much of an LLM's behavior can be changed by modifying a small number of low-rank directions in its pretrained weights?

---

## Mental Model

$$
\boxed{W_{\text{adapted}}=W_{\text{frozen}}+\frac{\alpha}{r}BA}
$$

- **Motivation:** Full fine-tuning is expensive.
- **Mechanism:** Learn low-rank updates instead of updating all pretrained weights.
- **Rank:** Controls the expressiveness of each weight update.
- **Placement:** Determines which computations can be modified.
- **Training:** Uses gradient descent to learn the adapter parameters.
- **Evaluation:** Determines whether the adaptation produces the intended behavioral change.

**The bigger idea:** LoRA is not just a memory-saving technique. It also provides a controlled way to investigate how small parameter changes influence the behavior of large pretrained neural networks.